# Numerical Methods Week, Notebook 3: Optimization

We now want to **minimise** a function rather than evaluate, integrate or interpolate one:

$$\min_{x \in C} f(x)$$

where $C$ is the set of allowed points. Everything here is a *first-order* method: the
only thing we ever ask of $f$ is its value and its gradient $\nabla f(x)$, the vector of
partial derivatives pointing in the direction of steepest **increase**. So $-\nabla f(x)$
points downhill, and every method below is a different answer to one question:

> **I know which way is downhill. How far do I go, and what do I do when the constraints
> say I cannot go there?**

Three exercises:

1. **Unconstrained gradient descent**: take a step downhill. Repeat. ($C = \mathbb{R}^n$.)
2. **Projected gradient descent**: step downhill, then snap back into $C$.
3. **Frank–Wolfe**: never leave $C$ at all: find the best point in $C$ by a *linear*
   approximation, and move partway toward it.

## The test problem

We use the same convex quadratic throughout, because everything about it is known
exactly and you can check every claim:

$$f(x) = \tfrac{1}{2}x^\top A x - b^\top x, \qquad \nabla f(x) = Ax - b$$

With $A$ symmetric positive definite this is a bowl, and the unconstrained minimiser is
the solution of $Ax = b$, namely $x^\star = A^{-1}b$. Having $x^\star$ in closed form is
what lets you *measure* convergence instead of guessing at it.

Two numbers about $A$ govern how the methods behave, and they will come up constantly:

- $L = \lambda_{\max}(A)$: the curvature in the steepest direction. The step size must
  satisfy $\alpha < 2/L$ or gradient descent diverges.
- $\kappa = \lambda_{\max}/\lambda_{\min}$: the **condition number**. $\kappa = 1$ is a
  perfectly round bowl and converges instantly; large $\kappa$ is a long narrow valley,
  and that is where first-order methods suffer.

Run the setup cell, then work through in order.

In [ ]:
# Setup: run this first.
# Import numpy as np and matplotlib.pyplot as plt.

---
# Exercise 1: Unconstrained gradient descent

## The method

$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

That is the whole algorithm. Its behaviour is governed entirely by $\alpha$, the **step
size** (in machine learning, the *learning rate*):

| $\alpha$ | what happens |
|---|---|
| too small | converges, but crawls; thousands of iterations |
| about right | steady geometric decrease |
| $\alpha > 2/L$ | **diverges**, oscillating with growing amplitude |

For our quadratic the threshold is exact and checkable: $\alpha < 2/\lambda_{\max}(A)$.
For a general $f$ you either use a line search or you tune it.

## Stopping

Stop when $\|\nabla f(x_k)\|$ is small; at a minimum the gradient is zero, so a small
gradient means you are nearly stationary. As always, also cap the iteration count.

## What it is not

Gradient descent finds a **local** minimum, and only that. On a convex $f$ (like ours)
local means global, so there is nothing to worry about. On a non-convex $f$ (every neural
network ever trained) where you start decides where you end up.

In [ ]:
# Step 1.1: Set up the problem.
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
b = np.array([1.0, -1.0])
# 1. Write  f(x)  and  grad_f(x)  for f(x) = 0.5 x^T A x - b^T x .
#    Hint: x @ A @ x for the quadratic form; A @ x - b for the gradient.
# 2. Compute the exact minimiser with np.linalg.solve(A, b) and call it x_star.
# 3. Print x_star, f(x_star), and grad_f(x_star): the gradient must be ~0.
# 4. Print the eigenvalues with np.linalg.eigvalsh(A), then compute L, kappa,
#    and the divergence threshold 2 / L.

In [ ]:
# Step 1.2: Draw the landscape.
# Make a contour plot of f over [-2, 2] x [-2, 2]:
#   - build a grid with np.meshgrid
#   - evaluate f at every grid point (a loop is fine here, or use the explicit
#     quadratic formula written out in terms of X and Y)
#   - plt.contour with ~25 levels, then mark x_star with a red star
# Keep this cell handy: you will overlay optimisation paths on it repeatedly.

In [ ]:
# Step 1.3: One step, by hand.
# Start from x0 = np.array([1.5, 1.5]) with alpha = 0.1 .
# Compute a single gradient step explicitly and print: x0, grad_f(x0), x1,
# and f(x0) versus f(x1). Did the function value actually go down?

In [ ]:
# Step 1.4: The loop.
# Write  gradient_descent(grad, x0, alpha, tol=1e-8, max_iter=1000)  returning
# (x, path) where path is an array of shape (n_iters + 1, 2) holding every iterate.
# Stop when np.linalg.norm(grad(x)) < tol, or at max_iter.
# Run it from x0 = [1.5, 1.5] with alpha = 0.1 and print the final x, the number of
# iterations, and the distance to x_star.

In [ ]:
# Step 1.5: Watch it walk.
# Re-draw the contour plot and overlay path[:, 0] vs path[:, 1] with 'o-'.
# Look at the corners: consecutive steps meet at right angles for this method.
# (Optional: check that numerically by taking the dot product of successive step
# directions; for exact line search it is exactly zero.)

In [ ]:
# Step 1.6: The step size sweep. (The important experiment.)
# Run the same problem for alpha in [0.01, 0.1, 0.3, 0.5, 0.62, 0.7] .
# On ONE figure, plot  f(x_k) - f(x_star)  against k with plt.semilogy, one line per
# alpha, with a legend.
# Compare the results with the threshold 2/L you computed in 1.1: which alphas
# converge, which diverges, and does the boundary land where the theory says?
# Guard against overflow: cap max_iter and use np.isfinite so a diverging run
# does not fill your notebook with nan warnings.

In [ ]:
# Step 1.7: Ill conditioning.
A_bad = np.array([[1.0, 0.0],
                  [0.0, 50.0]])
b_bad = np.array([1.0, 1.0])
# Repeat the experiment with this A: kappa is now 50, a long narrow valley.
# Use the largest stable alpha you can (just under 2/L) and plot the path on the
# contours of the new f.
# How many iterations does it need compared with 1.4? Describe the zig-zag in a comment,
# and explain which eigenvalue is limiting the step size and which one is limiting
# the progress.

In [ ]:
# Step 1.8: Backtracking line search.
# A fixed alpha is a guess. Instead, at each iteration start with alpha = 1.0 and halve
# it until the Armijo condition holds:
#     f(x - alpha * g)  <=  f(x) - 0.5 * alpha * (g @ g)      with g = grad_f(x)
# Add this to your solver and re-run the ill-conditioned problem from 1.7.
# Compare the iteration count with the fixed-step version. Was the tuning worth it?

In [ ]:
# Step 1.9: A real application: least squares by gradient descent.
rng = np.random.default_rng(0)
x_data = rng.uniform(0, 10, 50)
y_data = 2.5 * x_data - 1.0 + rng.normal(0, 1.0, 50)
# Fit a line y = m*x + c by minimising the mean squared error over theta = [m, c].
# 1. Write the loss and its gradient (work them out by hand; it is a quadratic in theta,
#    so everything above applies).
# 2. Minimise it with your gradient_descent.
# 3. Compare with the exact solution from np.linalg.lstsq.
# 4. Plot the data, your fitted line, and the true line y = 2.5x - 1.

In [ ]:
# Step 1.10: Stretch: let torch compute the gradient.
# Rewrite step 1.9 using torch: make theta a tensor with requires_grad=True, compute
# the loss, call loss.backward(), and use theta.grad in place of your hand-derived
# gradient (remember to zero it each iteration, and to wrap the update in
# `with torch.no_grad():`).
# Confirm you get the same fit. This is precisely the training loop from the ecosystem
# notebook; gradient descent is all it ever was.

---
# Exercise 2: Projected gradient descent

## The problem

Now the answer must lie in a set: $\min_{x \in C} f(x)$ with $C$ convex. A plain gradient
step will happily walk straight out of $C$, so we drag it back:

$$x_{k+1} = P_C\big(x_k - \alpha \nabla f(x_k)\big)$$

## Projection

$P_C(y)$ is the **closest point of $C$ to $y$**:

$$P_C(y) = \arg\min_{z \in C} \|z - y\|_2$$

For a convex $C$ this point exists and is unique. For the sets we care about it has a
closed form:

| Set $C$ | Projection $P_C(y)$ |
|---|---|
| box $\{l \le x \le u\}$ | clip each coordinate into $[l_i, u_i]$ |
| non-negative orthant $\{x \ge 0\}$ | $\max(y, 0)$ |
| $\ell_2$ ball $\{\|x\|_2 \le r\}$ | $y$ if $\|y\| \le r$, else $r\,y/\|y\|$ |
| hyperplane, simplex, ... | closed forms exist, more work |

Two properties every projection has, and both make excellent unit tests:

1. **Idempotence**: $P_C(P_C(y)) = P_C(y)$. Projecting twice changes nothing.
2. **Fixed points**: if $y \in C$ then $P_C(y) = y$. Points already inside do not move.

## What the answer looks like

When the unconstrained minimum lies inside $C$, the constraint does nothing and you get
the same answer as Exercise 1. When it lies outside, the solution sits **on the boundary**
— pinned against the constraint, with the negative gradient pushing outward. The
generalisation of "$\nabla f = 0$" is

$$x^\star = P_C\big(x^\star - \alpha \nabla f(x^\star)\big)$$

i.e. $x^\star$ is a *fixed point* of the iteration. That is directly checkable in code,
and step 2.7 asks you to check it.

## The catch

This only works when you can actually compute $P_C$. For a box, it is one call to
`np.clip`. For many interesting sets it is its own hard optimisation problem; which is
exactly the opening Frank–Wolfe exploits in Exercise 3.

In [ ]:
# Step 2.1: Write two projections.
# Write:
#   project_box(y, lo, hi)   -> clip y into the box (np.clip does it in one call)
#   project_ball(y, r)       -> y if ||y|| <= r, else r * y / ||y||
# Keep them pure: they must return a NEW array and never modify their input.

In [ ]:
# Step 2.2: Test the projection properties.
# For random points y (some inside C, some far outside), verify for BOTH projections:
#   (a) the result is inside C (within a tiny tolerance)
#   (b) idempotence: P(P(y)) == P(y)
#   (c) points already inside are unchanged
# Use np.allclose and print a clear pass/fail line for each property.

In [ ]:
# Step 2.3: Test that it really is the CLOSEST point.
# For a single y outside the unit ball, sample 20000 random points inside the ball
# and confirm that none of them is closer to y than project_ball(y, 1.0) is.
# Hint: sample directions with rng.normal, normalise them, and scale by
# rng.uniform(0, 1) ** (1 / d) to fill the ball. Print the minimum sampled distance
# alongside the projected distance.

In [ ]:
# Step 2.4: The algorithm.
# Write  projected_gradient_descent(grad, project, x0, alpha, tol=1e-10, max_iter=5000)
# returning (x, path). It is your gradient_descent with ONE extra call per iteration.
# For the stopping test, do NOT use ||grad||; at a constrained solution the gradient
# is not zero. Stop when the iterate stops moving:  ||x_{k+1} - x_k|| < tol .

In [ ]:
# Step 2.5: Constraint that does nothing.
# Use the original A, b from 1.1 and a box with lo = [-5, -5], hi = [5, 5], which
# contains x_star. Run PGD and confirm you recover the unconstrained answer from 1.4.
# This is the sanity check: a slack constraint must change nothing.

In [ ]:
# Step 2.6: Constraint that bites.
# Now use  lo = [-5, -5],  hi = [0.2, 0.2] , which excludes x_star.
# Run PGD, plot the path over the contours, and draw the box (plt.plot of its four
# corners, or plt.axvline / plt.axhline).
# Print the solution and check whether it lies ON the boundary. Which coordinate(s)
# are pinned?

In [ ]:
# Step 2.7: Verify the fixed-point condition.
# Take your constrained solution x_c from 2.6 and check
#     x_c  ==  project(x_c - alpha * grad_f(x_c))
# for a few different alphas. Print the residual norm for each.
# This is the constrained analogue of "the gradient is zero", and it should hold to
# solver tolerance regardless of alpha.

In [ ]:
# Step 2.8: A ball constraint.
# Minimise the same f subject to ||x||_2 <= 0.3 (a ball that excludes x_star).
# Run PGD with project_ball, plot the path with the circle drawn on top
# (use np.cos/np.sin over a fine angle grid to draw it).
# Confirm the solution lands on the circle: print ||x|| and compare with 0.3.
# Then check that -grad_f(x) points radially OUTWARD there, i.e. the cosine between
# -grad_f(x) and x is close to +1. Explain in a comment why it has to.

In [ ]:
# Step 2.9: Non-negative least squares.
rng = np.random.default_rng(1)
M_mat = rng.normal(size=(30, 5))
true_w = np.array([2.0, 0.0, 1.5, 0.0, 0.7])
obs = M_mat @ true_w + rng.normal(0, 0.1, 30)
# Fit w by minimising ||M_mat @ w - obs||^2 SUBJECT TO w >= 0.
# 1. Write the gradient:  2 * M_mat.T @ (M_mat @ w - obs) .
# 2. Choose a safe alpha from the largest eigenvalue of  2 * M_mat.T @ M_mat .
# 3. Solve it with PGD and the non-negative projection, np.maximum(w, 0).
# 4. Compare with the UNCONSTRAINED least-squares solution from np.linalg.lstsq:
#    does it have negative entries? Which fit is closer to true_w, and why is
#    imposing a constraint you know to be true worth doing?

In [ ]:
# Step 2.10: Stretch: projection onto the probability simplex.
# C = { x : x >= 0,  sum(x) == 1 } . The projection has a neat algorithm:
#   1. sort y descending into u
#   2. find rho = the largest index j (1-based) with  u[j-1] + (1 - u[:j].sum()) / j > 0
#   3. theta = (1 - u[:rho].sum()) / rho
#   4. return np.maximum(y + theta, 0)
# Implement it, test the three properties from 2.2, and confirm the output sums to 1.
# You will use this set again in the next exercise.

---
# Exercise 3: Vanilla Frank–Wolfe (conditional gradient)

## The trick

Projected gradient descent needs $P_C$. Frank–Wolfe needs something different and often
much cheaper: the ability to **minimise a linear function over $C$**.

At $x_k$, replace $f$ by its first-order Taylor expansion and minimise *that* over the
whole feasible set:

$$s_k = \arg\min_{s \in C} \; \langle \nabla f(x_k),\, s \rangle$$

This is the **linear minimisation oracle** (LMO). Then step partway toward $s_k$:

$$x_{k+1} = (1 - \gamma_k)\,x_k + \gamma_k s_k, \qquad \gamma_k = \frac{2}{k+2}$$

## Why this is clever

- **Feasibility is free.** $x_{k+1}$ is a convex combination of two points of $C$, so it
  is in $C$ automatically. No projection, ever.
- **The oracle is cheap on the sets that matter.** Minimising a linear function over a
  polytope always has a **vertex** as its answer, so the LMO usually reduces to "find the
  largest coordinate of the gradient":

| Set $C$ | $\arg\min_{s \in C}\langle g, s\rangle$ |
|---|---|
| $\ell_1$ ball of radius $r$ | $s = -r\,\mathrm{sign}(g_i)\,e_i$ where $i = \arg\max_j |g_j|$ |
| probability simplex | $s = e_i$ where $i = \arg\min_j g_j$ |
| $\ell_2$ ball of radius $r$ | $s = -r\,g/\|g\|$ |

- **The iterates are sparse.** After $k$ steps, $x_k$ is a convex combination of at most
  $k+1$ vertices. For the $\ell_1$ ball or the simplex that means **at most $k+1$ nonzero
  entries**; you get a sparse solution for free, which is why the method is popular in
  machine learning.

## The duality gap: a free stopping certificate

Define

$$g_k = \langle \nabla f(x_k),\, x_k - s_k \rangle$$

By convexity, $f(x_k) - f(x^\star) \le g_k$. This is remarkable: $g_k$ is computed from
quantities you already have, it costs nothing, and it is a **guaranteed upper bound on
how suboptimal you are**; without knowing $f(x^\star)$. Stop when $g_k < \varepsilon$
and you have a certificate, not a hope.

## The price

Convergence is $O(1/k)$: meaningfully slower than gradient descent's geometric rate on
nice problems. The zig-zag toward a boundary optimum is characteristic and is what the
"away step" and "pairwise" variants exist to fix. Vanilla Frank–Wolfe, the version here,
does not.

In [ ]:
# Step 3.1: The L1-ball oracle.
# Write  lmo_l1(g, r)  returning argmin over  ||s||_1 <= r  of  g @ s .
# The answer is a vertex: all zeros except a single entry at index argmax|g|,
# set to  -r * sign(g[i]) .
# Test it on g = [3, -7, 1] with r = 2: you should get [0, 2, 0]. Convince yourself
# by hand why that is the minimiser before moving on.

In [ ]:
# Step 3.2: Test the oracle against brute force.
# For random g in 2D and r = 1, compare  g @ lmo_l1(g, r)  with the smallest value of
# g @ s over 5000 random points sampled from the L1 ball.
# The oracle must be at least as good as every sample. Print both numbers.
# Hint: sample the L1 ball by drawing from the box and keeping points with
# np.abs(s).sum() <= r.

In [ ]:
# Step 3.3: The simplex oracle.
# Write  lmo_simplex(g)  returning e_i with i = argmin(g), the vertex of the
# probability simplex that minimises g @ s.
# Test it on g = [3, -7, 1]: you should get [0, 1, 0].

In [ ]:
# Step 3.4: The algorithm.
# Write  frank_wolfe(f, grad, lmo, x0, max_iter=200)  returning (x, path, gaps) where
# gaps[k] is the duality gap at iteration k. Each iteration:
#     g   = grad(x)
#     s   = lmo(g)
#     gap = g @ (x - s)                 # record this BEFORE stepping
#     gamma = 2 / (k + 2)
#     x   = x + gamma * (s - x)
# x0 must be feasible: for the L1 ball, the origin will do.

In [ ]:
# Step 3.5: Run it, and watch it stay feasible.
# Minimise the quadratic from 1.1 over the L1 ball of radius 0.5 (which excludes x_star).
# 1. Plot the path over the contours, and draw the L1 ball (a diamond: its vertices
#    are (r,0), (0,r), (-r,0), (0,-r)).
# 2. Print ||x_k||_1 at every 20th iteration; it must never exceed r.
# 3. Note the shape of the path: each step aims at a VERTEX, so it is polygonal, and
#    the steps get shorter as gamma = 2/(k+2) decays.

In [ ]:
# Step 3.6: The gap is a real bound.
# You need an accurate value of f at the constrained optimum. Get one either by
# running Frank-Wolfe for 20000 iterations, or with PGD from Exercise 2: projecting
# onto the L1 ball of radius r is exactly: if ||y||_1 <= r return y, otherwise project
# |y| onto the simplex scaled by r (your 2.10 code) and put the original signs back.
# Call the resulting value f_best.
# Then plot, on one semilogy figure:
#     the actual suboptimality  f(x_k) - f_best
#     the duality gap  gaps[k]
# Confirm the gap sits ABOVE the actual suboptimality at every iteration. That is the
# guarantee, verified.

In [ ]:
# Step 3.7: Measure the O(1/k) rate.
# Plot the duality gap against k on a log-log plot and fit the slope with np.polyfit
# on the tail (say k >= 20). You should get close to -1, i.e. gap ~ C/k.
# On the same axes, plot the suboptimality of gradient descent from Exercise 1 for
# comparison. Which is faster, and by how much after 200 iterations?

In [ ]:
# Step 3.8: Sparsity, the selling point.
# Run Frank-Wolfe on a 100-dimensional problem over the L1 ball:
rng = np.random.default_rng(3)
A_big = rng.normal(size=(100, 100))
A_big = A_big.T @ A_big + np.eye(100)     # symmetric positive definite
b_big = rng.normal(size=100)
# Start from the origin and run 50 iterations.
# After each iteration record how many entries of x are nonzero (use a small threshold,
# e.g. np.abs(x) > 1e-10) and plot that count against k.
# It should stay at or below k + 1. Explain in a comment why, using the fact that every
# iterate is a convex combination of vertices.
# Then compare with what PGD would give you; is ITS solution sparse?

In [ ]:
# Step 3.9: Over the simplex.
# Minimise the same 2D quadratic over the probability simplex using lmo_simplex,
# starting from x0 = [0.5, 0.5].
# Plot the path with the simplex drawn as the segment from (1,0) to (0,1).
# Confirm every iterate has non-negative entries summing to 1 (print the sums).

In [ ]:
# Step 3.10: Stretch: exact line search.
# The step size gamma = 2/(k+2) ignores the function entirely. For a quadratic you can
# minimise exactly along the segment. With d = s - x:
#     gamma* = clip( gap / (d @ A @ d), 0, 1 )
# Derive that on paper first (expand f(x + gamma*d) as a quadratic in gamma), then add
# it as an option to your solver and compare the gap curves against 2/(k+2) on the
# problem from 3.5. How much do you gain?

---
## What to take away

- **All three methods are the same idea.** Go downhill; differ only in how they handle
  the constraint set. Unconstrained: step freely. Projected: step, then snap back.
  Frank–Wolfe: never leave.
- **Step size is not a detail.** Too large diverges, too small crawls, and for a
  quadratic the threshold $2/\lambda_{\max}$ is exact and testable. When in doubt, use a
  line search rather than a magic number.
- **Conditioning decides the difficulty.** A large $\kappa$ means a narrow valley, and
  every first-order method zig-zags down it. This is the motivation for momentum,
  preconditioning, and everything that came after.
- **Constrained solutions live on the boundary**, where $\nabla f \ne 0$. The right
  optimality test is the fixed-point condition $x^\star = P_C(x^\star - \alpha \nabla
  f(x^\star))$, or a small Frank–Wolfe gap.
- **Choose the method by which subproblem is cheap.** Projection easy → PGD. Linear
  minimisation easy (or sparsity wanted) → Frank–Wolfe. That is the entire decision.
- **The duality gap is free accuracy information.** Most methods cannot tell you how
  suboptimal they are. Frank–Wolfe can, every iteration, at no cost.

You have now seen the same three ingredients all week: approximate a function locally
(Taylor, interpolation), estimate its derivatives (finite differences), and use those
estimates to drive an iteration (Newton, gradient descent). Every serious numerical method
you meet from here is a variation on that theme.